# Stage 00a — PatSeer Download

**What this notebook does:**  
Opens a Chrome browser, logs into PatSeer, and iterates through every record in
your search result.  For each patent it clicks the *Drawings* tab and downloads
all available figures — three file types that PatSeer provides:

| Type | Example filename | Meaning |
|------|-----------------|---------|
| `img` | `US20220267016A1_img003.png` | Individual figure crop — PatSeer already separated these, no splitting needed |
| `D`   | `US20220267016A1_D00005.png` | Full drawing sheet — may contain multiple figures per page |
| `FAT` | `US20220267016A1_FAT001.png` | Composite sheet — may contain multiple figures per page |

Raw PatSeer filenames (e.g. `record_0002_US…-20220825-img003.png`) are cleaned
on save: the `record_NNNN_` prefix and `-YYYYMMDD-` date segment are stripped,
remaining hyphens become underscores.

A `{patent_id}_download_manifest.json` is written into each patent folder
recording which files were saved, the publication date, and the record position.

**Where it fits in the pipeline:**
```
00a  ←  YOU ARE HERE  (download from PatSeer)
 ↓
00b  (positional matching → _F / _Fu labels)
 ↓
02   (pad + resize to 518×518)
```

**To resume a partial run:** set `START_FROM` in Cell 1 to the last completed
record index and re-run.  Already-downloaded patents are skipped automatically.

---

| Cell | What it does |
|------|--------------|
| 1 | Load config; set `OUTPUT_DIR`, `START_FROM`, `END_AT` |
| 2 | Launch the downloader — opens Chrome, prompts for login, then loops all records |
| 3 | Scan `raw/` and print a per-type download coverage report |

In [1]:
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

print(f"config_loader loaded from: {_cl_path}")

for p in [str(repo_root), str(src_dir)]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_dir))

cfg = load_config()

# ─── Batch mode ────────────────────────────────────────────────────────────────
# MISSING_BATCH = True  → download into raw/missing_batch/  (for the 296 missing families)
# MISSING_BATCH = False → download into raw/               (normal full-dataset run)
MISSING_BATCH = True

_raw = Path(cfg["paths"]["raw_images"])
OUTPUT_DIR = _raw / "missing_batch" if MISSING_BATCH else _raw
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Run parameters ────────────────────────────────────────────────────────────
# Paste your PatSeer search URL here (copy from browser address bar after running the PNC search)
SEARCH_BASE_URL = "https://app1.patseer.com/#!/result/ad056431-8522-11e8-944f-22000bd445e0/172"

_start_raw  = input("Start from record number [1]: ").strip()
START_FROM  = int(_start_raw) if _start_raw.isdigit() else 1

# Enter the number of results PatSeer found for your PNC query
# (will be different from 1639 — PatSeer may find fewer if some pub numbers aren't indexed)
_total_raw  = input("Total records in THIS PatSeer search: ").strip()
TOTAL_RECORDS = int(_total_raw) if _total_raw.isdigit() else 1

END_AT = None  # set to int to stop early; None = run to TOTAL_RECORDS
# ──────────────────────────────────────────────────────────────────────────────

print(f"Batch mode     : {'MISSING (→ raw/missing_batch/)' if MISSING_BATCH else 'NORMAL (→ raw/)'}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Start from     : {START_FROM}")
print(f"Total records  : {TOTAL_RECORDS}")
print(f"End at         : {END_AT or TOTAL_RECORDS}")


config_loader loaded from: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src/config_loader.py
Batch mode     : MISSING (→ raw/missing_batch/)
Output dir     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw/missing_batch
Start from     : 1
Total records  : 299
End at         : 299


In [2]:
# ── Rename record_XXXX folders → real patent ID using Excel lookup ────────────
# IMPORTANT: This only works correctly if the PatSeer download AND the Excel
# export used the SAME sort order.  Always use 'Document No ↑' for both.
# If sort orders differ, record_position will map to the wrong patent.
#
# For existing record_ folders from an unknown sort run, use PNC re-download
# instead: the audit notebook generates the PNC query for missing patents.
#
# Safe to re-run: already-renamed folders are skipped automatically.

import re, json
import pandas as pd

EXCEL_PATH = Path(cfg["paths"]["patseer_excel"])
raw_dir    = Path(cfg["paths"]["raw_images"])

if not raw_dir.exists():
    print("raw/ folder does not exist yet — nothing to migrate.")
elif not EXCEL_PATH.exists():
    print(f"Excel not found: {EXCEL_PATH}")
else:
    df_excel = pd.read_excel(EXCEL_PATH, usecols=[0])
    print(f"Excel loaded: {len(df_excel)} rows  |  scanning for record_XXXX folders…")
    print("⚠  Only run this if the download and Excel export both used 'Document No ↑' sort.\n")

    migrated, skipped, conflicts = [], [], []

    for folder in sorted(raw_dir.iterdir()):
        if not folder.is_dir() or not re.match(r"record_\d+$", folder.name):
            continue

        manifests = list(folder.glob("*manifest*.json"))
        rec_pos   = None
        if manifests:
            try:
                data    = json.loads(manifests[0].read_text())
                rec_pos = data.get("record_position")
            except Exception:
                pass

        patent_id = None
        if rec_pos is not None:
            row_idx = int(rec_pos) - 1
            if 0 <= row_idx < len(df_excel):
                val = str(df_excel.iloc[row_idx, 0]).strip()
                if val and val.lower() not in ("nan", "none", ""):
                    patent_id = val

        if not patent_id:
            print(f"  ? {folder.name}  (rec_pos={rec_pos}) — not in Excel, skipping")
            skipped.append(folder.name)
            continue

        dest_folder = raw_dir / patent_id
        if dest_folder.exists():
            # Collision: a folder with this patent ID was already downloaded properly.
            # The record_ folder's images are likely duplicates or from wrong sort order.
            existing_imgs = len(list(dest_folder.glob("*.png")))
            record_imgs   = len(list(folder.glob("*.png")))
            print(f"  ⚠ {folder.name} → {patent_id} already exists "
                  f"(record has {record_imgs} imgs, existing has {existing_imgs} imgs) — skipping")
            conflicts.append((folder.name, patent_id))
            continue

        record_prefix = folder.name + "_"
        for f in list(folder.glob("*")):
            if f.name.startswith(record_prefix):
                f.rename(folder / f.name[len(record_prefix):])

        mf_path = folder / "download_manifest.json"
        if mf_path.exists():
            mf_data = json.loads(mf_path.read_text())
            mf_data["patent_id"] = patent_id
            for key in ("img_files", "d_files", "fat_files"):
                mf_data[key] = [
                    n[len(record_prefix):] if n.startswith(record_prefix) else n
                    for n in mf_data.get(key, [])
                ]
            (folder / f"{patent_id}_download_manifest.json").write_text(
                json.dumps(mf_data, indent=2))
            mf_path.unlink(missing_ok=True)

        folder.rename(dest_folder)
        print(f"  ✓ {folder.name}  →  {patent_id}  (Excel row {rec_pos})")
        migrated.append((folder.name, patent_id))

    print(f"\nMigrated : {len(migrated)}")
    print(f"Conflicts: {len(conflicts)}  ← these record_ folders may have wrong patent ID")
    print(f"Unknown  : {len(skipped)}")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Excel loaded: 1639 rows  |  scanning for record_XXXX folders…
⚠  Only run this if the download and Excel export both used 'Document No ↑' sort.


Migrated : 0
Conflicts: 0  ← these record_ folders may have wrong patent ID
Unknown  : 0


In [ ]:
import patseer_downloader as dl

# Patch module-level configuration before launching
dl.OUTPUT_DIR     = OUTPUT_DIR
dl.START_FROM     = START_FROM
dl.TOTAL_RECORDS  = END_AT if END_AT is not None else TOTAL_RECORDS
dl.EXCEL_PATH       = Path(cfg["paths"]["patseer_excel"])  # used to resolve record_ folders
dl.SEARCH_BASE_URL  = SEARCH_BASE_URL

# A Chrome window will open — log in to PatSeer, then press Enter here.
# clean_filename() is applied to every saved file inside download_images().
dl.main()



  Chrome opened. Loading your PatSeer search…

  ⚠  Login page detected — please log in in the Chrome window.
  ⚠  Login page detected — please log in in the Chrome window.
  ⚠  Timeout — proceeding anyway.
  Starting from record 1…


[  1/299]
  Patent : IN202011031027A
  Images : 0 found
  ⚠  No image URLs collected.

[  2/299]
  Patent : DE102024127443A1
  Images : 3 found
      ✓ DE102024127443A1_fig_01.png  (16 kB)
      ✓ DE102024127443A1_fig_02.png  (104 kB)
      ✓ DE102024127443A1_fig_03.png  (104 kB)

[  3/299]
  Patent : DE102024122189A1
  Images : 11 found
      ✓ DE102024122189A1_fig_01.png  (18 kB)
      ✓ DE102024122189A1_fig_02.png  (80 kB)
      ✓ DE102024122189A1_fig_03.png  (98 kB)
      ✓ DE102024122189A1_fig_04.png  (100 kB)
      ✓ DE102024122189A1_fig_05.png  (91 kB)
      ✓ DE102024122189A1_fig_06.png  (112 kB)
      ✓ DE102024122189A1_fig_07.png  (100 kB)
      ✓ DE102024122189A1_fig_08.png  (74 kB)
      ✓ DE102024122189A1_fig_09.png  (82 kB)
      ✓ DE102024

In [ ]:
import re

raw_dir = OUTPUT_DIR

patents_attempted  = 0
patents_with_img   = 0
patents_with_d     = 0
patents_with_fat   = 0
total_img          = 0
total_d            = 0
total_fat          = 0

for patent_dir in sorted(raw_dir.iterdir()):
    if not patent_dir.is_dir():
        continue
    patents_attempted += 1

    imgs  = [p for p in patent_dir.glob("*.png") if "_img" in p.name.lower()]
    dfiles = [p for p in patent_dir.glob("*.png") if re.search(r"_d\d{4,}", p.name, re.IGNORECASE)]
    fats  = [p for p in patent_dir.glob("*.png") if "_fat" in p.name.lower()]

    if imgs:   patents_with_img += 1
    if dfiles: patents_with_d   += 1
    if fats:   patents_with_fat += 1

    total_img += len(imgs)
    total_d   += len(dfiles)
    total_fat += len(fats)

def _pct(n):
    return f"{n / patents_attempted * 100:.0f}%" if patents_attempted else "N/A"

avg_imgs = total_img / patents_with_img if patents_with_img else 0

print("══════════════════════════════════════")
print("Download Coverage Report")
print("══════════════════════════════════════")
print(f"Patents attempted      : {patents_attempted}")
print(f"Patents with img files : {patents_with_img}  ({_pct(patents_with_img)})")
print(f"Patents with D files   : {patents_with_d}  ({_pct(patents_with_d)})")
print(f"Patents with FAT files : {patents_with_fat}  ({_pct(patents_with_fat)})")
print(f"Total img files        : {total_img}")
print(f"Total D files          : {total_d}")
print(f"Total FAT files        : {total_fat}")
print(f"Average imgs/patent    : {avg_imgs:.1f}")
print("══════════════════════════════════════")